# Synthetic Learner Dataset Generator

**Problem:** research teams need safe, realistic test data without using real student records.

**Setting:** students sit an exam (online or physical). The exam score decides **Pass / Fail**.

**What this notebook does (run the cells from top to bottom):**

1. **Schema design:** the rules every record must follow, plus a validator that lists problems and suggested fixes
2. **Data generators:** a *rule-based* generator and a *seed-based* generator (learns from your hand-made rows)
3. **Quality metrics:** schema checks, side-by-side statistics, and a usefulness test (can a model predict Pass/Fail?)
4. **Privacy checks:** duplicates, seed copies, near-copies, uniqueness, and identifier scan
5. **Tests:** automatic checks that the core flows work

**Main challenge:** judge realism *without* copying real student records. We do it by checking that the patterns we designed show up, that a model can learn from the data, and that the two independent generators agree on the main drivers.

*Needs:* `numpy`, `pandas`, `scikit-learn`, `matplotlib` (`pip install numpy pandas scikit-learn matplotlib`).

## 0. Setup and settings
Every decision you may need to explain lives in the settings below.

In [ ]:
import unittest
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ---- Settings ----
RANDOM_SEED = 42        # same seed = same data every time (reproducibility)
N_ROWS = 500            # how many fake students to generate
PASS_MARK = 50          # exam_score >= 50 means Pass
SEED_FILE = "my_seed.csv"   # your hand-made seed rows

COLUMNS = ["ID", "Age", "Study_hrs", "Attendance%", "Total_screen_time",
           "Sleep_hrs", "Internet_quality", "exam_score", "Result"]
ALL_NUM = ["Age", "Study_hrs", "Attendance%", "Total_screen_time",
           "Sleep_hrs", "exam_score"]
# exam_score is NOT a model feature: Result is calculated from it (data leakage)
FEATURE_NUM = ["Age", "Study_hrs", "Attendance%", "Total_screen_time", "Sleep_hrs"]
FEATURE_CAT = ["Internet_quality"]
FEATURES = FEATURE_NUM + FEATURE_CAT

print("Setup complete")

## 1. Schema design

The schema is the blueprint: what each column is and what values are allowed.
`validate_schema` checks a dataset against it and returns an **issue list with a suggested fix** for each problem.

In [ ]:
SCHEMA = {
    "ID":                {"type": "id", "pattern": r"^S\d{4,}$"},
    "Age":               {"type": "number", "min": 17, "max": 45},
    "Study_hrs":         {"type": "number", "min": 0, "max": 40},
    "Attendance%":       {"type": "number", "min": 0, "max": 100},
    "Total_screen_time": {"type": "number", "min": 0, "max": 24},
    "Sleep_hrs":         {"type": "number", "min": 0, "max": 24},
    "Internet_quality":  {"type": "category", "allowed": ["Poor", "Average", "Good"]},
    "exam_score":        {"type": "number", "min": 0, "max": 100},
    "Result":            {"type": "category", "allowed": ["Pass", "Fail"]},
}

# Exception thresholds: a pass rate outside this range looks unrealistic
THRESHOLDS = {"pass_rate_min": 0.30, "pass_rate_max": 0.90}


def internet_label(values):
    """Turn 1-5 ratings into labels: 1-2 Poor, 3 Average, 4-5 Good."""
    return pd.cut(values, bins=[0, 2, 3, 5],
                  labels=["Poor", "Average", "Good"]).astype(str)


def standardise(df):
    """Make column names and internet labels match the schema."""
    df = df.copy()
    lookup = {c.lower(): c for c in COLUMNS}
    df.columns = [lookup.get(c.strip().replace(" ", "_").lower(), c.strip())
                  for c in df.columns]
    q = df["Internet_quality"]
    if pd.api.types.is_numeric_dtype(q):
        df["Internet_quality"] = internet_label(q)
    else:
        df["Internet_quality"] = q.astype(str).str.strip().str.title()
    return df


def validate_schema(df):
    """Return a list of (problem, suggested_fix). An empty list means clean."""
    issues = []
    missing_cols = [c for c in COLUMNS if c not in df.columns]
    if missing_cols:
        return [(f"Missing columns: {missing_cols}",
                 "Add them or fix the spelling to match the schema.")]
    extra = [c for c in df.columns if c not in COLUMNS]
    if extra:
        issues.append((f"Unexpected columns: {extra}",
                       "Remove them or add them to SCHEMA."))

    n_missing = int(df[COLUMNS].isna().sum().sum())
    if n_missing:
        issues.append((f"{n_missing} missing values",
                       "Fill them in or drop those rows."))

    for col, rule in SCHEMA.items():
        s = df[col]
        if rule["type"] == "number":
            num = pd.to_numeric(s, errors="coerce")
            not_num = int(num.isna().sum() - s.isna().sum())
            if not_num:
                issues.append((f"{col}: {not_num} values are not numbers",
                               "Replace text with numbers."))
            out = int(((num < rule["min"]) | (num > rule["max"])).sum())
            if out:
                issues.append((f"{col}: {out} values outside {rule['min']} to {rule['max']}",
                               "Clip the values or regenerate those rows."))
        elif rule["type"] == "category":
            bad = int((~s.isin(rule["allowed"]) & s.notna()).sum())
            if bad:
                issues.append((f"{col}: {bad} values not in {rule['allowed']}",
                               "Correct the labels (check spelling and capital letters)."))
        elif rule["type"] == "id":
            bad = int((~s.astype(str).str.match(rule["pattern"])).sum())
            if bad:
                issues.append((f"{col}: {bad} IDs do not look like fake IDs",
                               "Replace them with generated IDs like S0001."))

    dup_ids = int(df["ID"].duplicated().sum())
    if dup_ids:
        issues.append((f"{dup_ids} duplicate IDs", "Regenerate IDs so each is unique."))
    dup_rows = int(df.drop(columns=["ID"]).duplicated().sum())
    if dup_rows:
        issues.append((f"{dup_rows} duplicate rows (ignoring ID)",
                       "Use drop_duplicates() or regenerate with another seed."))

    score = pd.to_numeric(df["exam_score"], errors="coerce")
    expected = np.where(score >= PASS_MARK, "Pass", "Fail")
    mismatch = int((df["Result"].to_numpy() != expected).sum())
    if mismatch:
        issues.append((f"{mismatch} rows where Result disagrees with the {PASS_MARK}-mark threshold",
                       "Recalculate Result from exam_score."))

    pass_rate = (df["Result"] == "Pass").mean()
    if not THRESHOLDS["pass_rate_min"] <= pass_rate <= THRESHOLDS["pass_rate_max"]:
        issues.append((f"Pass rate {pass_rate:.0%} is outside the expected range",
                       "Adjust the generator settings so the pass rate looks realistic."))
    return issues


def show_issues(name, issues):
    if not issues:
        print(f"{name}: no issues found")
        return
    print(f"{name}: {len(issues)} issue(s)")
    for i, (problem, fix) in enumerate(issues, 1):
        print(f"  {i}. {problem}\n     Suggested fix: {fix}")

print("Schema ready")

## 2a. Generator A: rule-based

Patterns are **rules you choose** (study more = higher score, and so on). Each rule is a number in `CONFIG`, so you can explain it and change it.

In [ ]:
CONFIG = {
    "score_noise": 8,           # randomness in scores: real life is never exact
    "weights": {
        "base": 17,
        "study_hrs": 1.3,       # each weekly study hour adds ~1.3 marks
        "attendance": 0.30,     # each attendance % adds ~0.3 marks
        "sleep_penalty": 1.5,   # lose marks the further sleep is from 7.5 hrs
        "screen_time": 0.9,     # each daily screen hour above 6 costs ~0.9 marks
        "internet": 1.5,        # each internet step (1-5) adds ~1.5 marks
    },
}


def generate_rule_based(n_rows=N_ROWS, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    w = CONFIG["weights"]

    ids = [f"S{i:04d}" for i in range(1, n_rows + 1)]
    age = np.clip(rng.normal(22, 4, n_rows), 17, 45).round().astype(int)
    study = np.clip(rng.normal(14, 6, n_rows), 0, 40).round(1)
    attendance = np.clip(70 + 0.6 * (study - 14) + rng.normal(0, 10, n_rows),
                         30, 100).round(1)
    screen = np.clip(rng.normal(7, 2.5, n_rows), 1, 16).round(1)
    sleep = np.clip(rng.normal(6.8, 1.2, n_rows), 3, 10).round(1)
    internet = rng.choice([1, 2, 3, 4, 5], size=n_rows, p=[0.10, 0.20, 0.30, 0.25, 0.15])

    score = (w["base"]
             + w["study_hrs"] * study
             + w["attendance"] * attendance
             - w["sleep_penalty"] * np.abs(sleep - 7.5)
             - w["screen_time"] * np.maximum(screen - 6, 0)
             + w["internet"] * internet
             + rng.normal(0, CONFIG["score_noise"], n_rows))
    score = np.clip(score, 0, 100).round(1)

    return pd.DataFrame({
        "ID": ids, "Age": age, "Study_hrs": study, "Attendance%": attendance,
        "Total_screen_time": screen, "Sleep_hrs": sleep,
        "Internet_quality": internet_label(pd.Series(internet)),
        "exam_score": score,
        "Result": np.where(score >= PASS_MARK, "Pass", "Fail"),
    })[COLUMNS]

print("Rule-based generator ready")

## 2b. Generator B: seed-based

Learns the **averages** and **relationships** from a small table you wrote by hand (`my_seed.csv`), then creates new rows that follow them.
`Internet_quality` is a category, so it is sampled separately using the label proportions in your seed rows.

If `my_seed.csv` is not found, a made-up demo seed is used so the notebook still runs.

In [ ]:
def make_demo_seed(n=25, seed=1):
    """A made-up seed table, only used if you have no seed file yet."""
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({
        "ID": [f"S{i:04d}" for i in range(1, n + 1)],
        "Age": rng.integers(18, 30, n),
        "Study_hrs": rng.integers(2, 25, n),
        "Attendance%": rng.integers(40, 100, n),
        "Total_screen_time": rng.integers(3, 12, n),
        "Sleep_hrs": rng.choice([5, 5.5, 6, 6.5, 7, 7.5, 8], n),
        "Internet_quality": rng.choice(["Poor", "Average", "Good"], n),
    })
    d["exam_score"] = (15 + 1.2 * d["Study_hrs"] + 0.35 * d["Attendance%"]
                       + rng.normal(0, 7, n)).clip(0, 100).round(1)
    d["Result"] = np.where(d["exam_score"] >= PASS_MARK, "Pass", "Fail")
    return d[COLUMNS]


def load_seed(path=SEED_FILE):
    if Path(path).exists():
        return standardise(pd.read_csv(path)), f"Using your seed file: {path}"
    return make_demo_seed(), f"'{path}' not found, using a made-up demo seed instead"


def generate_seed_based(seed_df, n_rows=N_ROWS, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    mean = seed_df[ALL_NUM].mean()      # average of each numeric column
    cov = seed_df[ALL_NUM].cov()        # how the columns move together
    df = pd.DataFrame(rng.multivariate_normal(mean, cov, size=n_rows), columns=ALL_NUM)

    # make values believable
    df["Age"] = df["Age"].clip(17, 45).round().astype(int)
    df["Study_hrs"] = df["Study_hrs"].clip(0, 40).round(1)
    df["Attendance%"] = df["Attendance%"].clip(30, 100).round(1)
    df["Total_screen_time"] = df["Total_screen_time"].clip(1, 16).round(1)
    df["Sleep_hrs"] = df["Sleep_hrs"].clip(3, 10).round(1)
    df["exam_score"] = df["exam_score"].clip(0, 100).round(1)

    # Internet quality: copy how often each label appears in the seed rows
    shares = seed_df["Internet_quality"].value_counts(normalize=True)
    df["Internet_quality"] = rng.choice(shares.index, size=n_rows, p=shares.values)

    df["Result"] = np.where(df["exam_score"] >= PASS_MARK, "Pass", "Fail")
    df.insert(0, "ID", [f"S{i:04d}" for i in range(1, n_rows + 1)])
    return df[COLUMNS]

print("Seed-based generator ready")

### Generate both datasets and save them

In [ ]:
rule_df = generate_rule_based(N_ROWS, RANDOM_SEED)
seed_df, seed_message = load_seed(SEED_FILE)
seed_based_df = generate_seed_based(seed_df, N_ROWS, RANDOM_SEED)

rule_df.to_csv("synthetic_students.csv", index=False)
seed_based_df.to_csv("my_synthetic_students.csv", index=False)

datasets = {"Rule-based": rule_df, "Seed-based": seed_based_df}

print(seed_message)
print("\nRule-based sample:")
display(rule_df.head())
print("Seed-based sample:")
display(seed_based_df.head())

## 3. Quality metrics

### 3a. Schema validation and summary statistics

In [ ]:
for name, df in datasets.items():
    show_issues(name, validate_schema(df))
    print()

summary = pd.DataFrame({name: df[ALL_NUM].mean().round(1) for name, df in datasets.items()})
summary.loc["Pass rate"] = [round((df["Result"] == "Pass").mean(), 2) for df in datasets.values()]
display(summary)

### 3b. Distributions side by side
Similar shapes suggest similar patterns; big gaps show where the generators disagree.

In [ ]:
cols = ["Study_hrs", "Attendance%", "Sleep_hrs", "exam_score"]
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, col in zip(axes.ravel(), cols):
    for name, df in datasets.items():
        ax.hist(df[col], bins=25, alpha=0.5, label=name)
    ax.set_title(col)
axes[0, 0].legend()
plt.tight_layout()
plt.show()

### 3c. Usefulness: can a model predict Pass/Fail?

If the data holds real patterns, a model trained on it should predict Pass/Fail well, tested with **5-fold cross-validation** (the `+/-` is the uncertainty).
Then a **cross-test**: train on one dataset, test on the other. High scores here mean the two datasets agree on the patterns.

*`exam_score` is not used as an input, otherwise the model would just learn the threshold and look perfect (data leakage).*

In [ ]:
def prepare(df):
    d = df.copy()
    d["target"] = (d["Result"] == "Pass").astype(int)
    return d


def build_model(kind="logistic"):
    prep = ColumnTransformer([
        ("num", StandardScaler(), FEATURE_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURE_CAT),
    ])
    if kind == "logistic":
        clf = LogisticRegression(max_iter=1000)
    else:
        clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED)
    return Pipeline([("prep", prep), ("clf", clf)])


def usefulness_table(data):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    rows = []
    for name, df in data.items():
        d = prepare(df)
        for kind in ("logistic", "forest"):
            res = cross_validate(build_model(kind), d[FEATURES], d["target"], cv=cv,
                                 scoring=["accuracy", "f1", "roc_auc"])
            row = {"dataset": name, "model": kind}
            for m in ("accuracy", "f1", "roc_auc"):
                s = res[f"test_{m}"]
                row[m] = f"{s.mean():.2f} +/- {s.std():.2f}"
            rows.append(row)
    return pd.DataFrame(rows)


def cross_test_table(data):
    names = list(data)
    rows = []
    for a, b in [(names[0], names[1]), (names[1], names[0])]:
        da, db = prepare(data[a]), prepare(data[b])
        model = build_model("logistic").fit(da[FEATURES], da["target"])
        pred = model.predict(db[FEATURES])
        proba = model.predict_proba(db[FEATURES])[:, 1]
        rows.append({"train on": a, "test on": b,
                     "accuracy": round(accuracy_score(db["target"], pred), 2),
                     "f1": round(f1_score(db["target"], pred), 2),
                     "roc_auc": round(roc_auc_score(db["target"], proba), 2)})
    return pd.DataFrame(rows)


def feature_influence(df):
    d = prepare(df)
    model = build_model("logistic").fit(d[FEATURES], d["target"])
    names = model.named_steps["prep"].get_feature_names_out()
    coefs = model.named_steps["clf"].coef_[0]
    out = pd.Series(coefs, index=names).round(2)
    return out.reindex(out.abs().sort_values(ascending=False).index)


print("5-fold cross-validation:")
display(usefulness_table(datasets))
print("\nCross-test (do the datasets agree on the patterns?):")
display(cross_test_table(datasets))
for name, df in datasets.items():
    print(f"\nFeature influence, {name} (positive = raises pass chance):")
    display(feature_influence(df))

## 4. Privacy checks

For each dataset:

1. Duplicate IDs and rows
2. **Exact copies** of seed rows
3. **Near-copies** of seed rows (nearest-neighbour distance in standardised units)
4. **Uniqueness**: how many students sit in tiny groups when described by age band, study-hours band and internet quality
5. **Direct identifiers**: emails, phone numbers, or IDs that do not look fake

The thresholds are at the top of the next cell. Say why you chose them in your README.

In [ ]:
NEAR_COPY_DISTANCE = 0.10   # closer than this (standardised units) = near-copy
MAX_NEAR_COPY_PCT = 0.0     # flag if ANY row is a near-copy of a seed row
K_MIN = 3                   # a row should look like at least 3 students
MAX_RISKY_PCT = 10.0        # flag if more than 10% of rows are in groups < K_MIN
EMAIL = r"[\w.+-]+@[\w-]+\.[\w.-]+"
PHONE = r"\+?\d[\d\s().-]{8,}\d"


def privacy_checks(df, seed, verbose=True):
    """Return a list of (problem, suggested_fix)."""
    issues = []
    say = print if verbose else (lambda *a, **k: None)

    # 1. duplicates
    dup_ids = int(df["ID"].duplicated().sum())
    dup_rows = int(df.drop(columns=["ID"]).duplicated().sum())
    say(f"  Duplicate IDs: {dup_ids} | Duplicate rows: {dup_rows}")
    if dup_ids:
        issues.append((f"{dup_ids} duplicate IDs", "Regenerate IDs so each is unique."))
    if dup_rows:
        issues.append((f"{dup_rows} duplicate rows", "Drop duplicates or regenerate with another seed."))

    # 2. exact copies of seed rows
    cols = ALL_NUM + ["Internet_quality"]
    left = df[cols].astype({c: float for c in ALL_NUM})
    right = seed[cols].astype({c: float for c in ALL_NUM}).drop_duplicates()
    exact = len(left.merge(right, on=cols, how="inner"))
    say(f"  Exact copies of seed rows: {exact}")
    if exact:
        issues.append((f"{exact} rows are exact copies of seed rows",
                       "Remove them and regenerate; add more random noise."))

    # 3. near-copies via nearest-neighbour distance
    both = pd.concat([df[ALL_NUM], seed[ALL_NUM]])
    std = both.std().replace(0, 1)
    a = ((df[ALL_NUM] - both.mean()) / std).to_numpy(dtype=float)
    b = ((seed[ALL_NUM] - both.mean()) / std).to_numpy(dtype=float)
    nearest = np.sqrt(((a[:, None, :] - b[None, :, :]) ** 2).sum(axis=2)).min(axis=1)
    near = int((nearest < NEAR_COPY_DISTANCE).sum())
    near_pct = 100 * near / len(df)
    say(f"  Nearest distance to a seed row: min {nearest.min():.2f}, median {np.median(nearest):.2f}")
    say(f"  Near-copies (distance < {NEAR_COPY_DISTANCE}): {near} rows ({near_pct:.1f}%)")
    if near_pct > MAX_NEAR_COPY_PCT:
        issues.append((f"{near} rows are near-copies of seed rows",
                       "Add more noise, use more seed rows, or remove those rows."))

    # 4. uniqueness / re-identification risk
    d = df.copy()
    d["age_band"] = pd.cut(d["Age"], [0, 19, 22, 25, 30, 120])
    d["study_band"] = pd.cut(d["Study_hrs"], list(range(0, 45, 5)), include_lowest=True)
    k = d.groupby(["age_band", "study_band", "Internet_quality"], observed=True)["ID"].transform("count")
    risky = int((k < K_MIN).sum())
    risky_pct = 100 * risky / len(d)
    say(f"  Rows in groups smaller than {K_MIN}: {risky} ({risky_pct:.1f}%), smallest group: {int(k.min())}")
    if risky_pct > MAX_RISKY_PCT:
        issues.append((f"{risky_pct:.1f}% of rows are in groups smaller than {K_MIN}",
                       "Use wider bands when sharing, or merge/remove rare combinations."))

    # 5. direct identifiers
    bad_ids = int((~df["ID"].astype(str).str.match(SCHEMA["ID"]["pattern"])).sum())
    found = 0
    for col in [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]:
        text = df[col].astype(str)
        found += int(text.str.contains(EMAIL, regex=True).sum())
        if col != "ID":
            found += int(text.str.contains(PHONE, regex=True).sum())
    say(f"  Bad-looking IDs: {bad_ids} | Emails/phone numbers: {found}")
    if bad_ids:
        issues.append((f"{bad_ids} IDs do not look like fake IDs", "Replace them with generated IDs."))
    if found:
        issues.append((f"{found} possible emails or phone numbers", "Delete or mask them."))
    return issues


all_privacy = {}
for name, df in datasets.items():
    print(f"{name}:")
    all_privacy[name] = privacy_checks(df, seed_df)
    print()

print("PRIVACY ISSUE LIST")
for name, issues in all_privacy.items():
    show_issues(name, issues)
print("\nNote: these checks reduce risk but cannot prove privacy. They depend on the thresholds above.")

## 5. Tests

Automatic checks of the core flows: schema, reproducibility, no leakage, and the privacy checks catching **planted** problems (so we know they work, not only that they stay quiet).

In [ ]:
class TestGenerators(unittest.TestCase):
    def setUp(self):
        self.seed = make_demo_seed()

    def test_rule_based_has_nine_columns_in_order(self):
        self.assertEqual(list(generate_rule_based(100, 1).columns), COLUMNS)

    def test_rule_based_same_seed_same_data(self):
        pd.testing.assert_frame_equal(generate_rule_based(200, 5), generate_rule_based(200, 5))

    def test_rule_based_different_seed_differs(self):
        self.assertFalse(generate_rule_based(200, 5).equals(generate_rule_based(200, 6)))

    def test_rule_based_passes_schema(self):
        self.assertEqual(validate_schema(generate_rule_based(300, 2)), [])

    def test_seed_based_passes_schema(self):
        self.assertEqual(validate_schema(generate_seed_based(self.seed, 300, 2)), [])

    def test_seed_based_reproducible(self):
        pd.testing.assert_frame_equal(generate_seed_based(self.seed, 200, 5),
                                      generate_seed_based(self.seed, 200, 5))

    def test_more_study_means_higher_scores(self):
        big = generate_rule_based(2000, 1)
        self.assertGreater(big["Study_hrs"].corr(big["exam_score"]), 0.4)

    def test_schema_validator_catches_bad_data(self):
        bad = generate_rule_based(50, 1)
        bad.loc[0, "Age"] = 200
        bad.loc[1, "Internet_quality"] = "Excellent"
        text = " ".join(p for p, _ in validate_schema(bad))
        self.assertIn("Age", text)
        self.assertIn("Internet_quality", text)


class TestEvaluation(unittest.TestCase):
    def test_exam_score_is_not_a_feature(self):
        for banned in ("exam_score", "Result", "ID", "target"):
            self.assertNotIn(banned, FEATURES)

    def test_model_beats_majority_guess(self):
        d = prepare(generate_rule_based(800, 5))
        train, test = d.iloc[:600], d.iloc[600:]
        model = build_model("logistic").fit(train[FEATURES], train["target"])
        acc = (model.predict(test[FEATURES]) == test["target"]).mean()
        majority = max(test["target"].mean(), 1 - test["target"].mean())
        self.assertGreater(acc, majority)


class TestPrivacy(unittest.TestCase):
    def setUp(self):
        self.seed = make_demo_seed()
        self.clean = generate_rule_based(500, 42)

    def problems(self, df):
        return " | ".join(p for p, _ in privacy_checks(df, self.seed, verbose=False))

    def test_clean_data_has_no_issues(self):
        self.assertEqual(self.problems(self.clean), "")

    def test_duplicate_id_caught(self):
        df = self.clean.copy()
        df.loc[2, "ID"] = df.loc[3, "ID"]
        self.assertIn("duplicate IDs", self.problems(df))

    def test_exact_seed_copy_caught(self):
        df = self.clean.copy()
        df[ALL_NUM] = df[ALL_NUM].astype(float)
        df.loc[0, ALL_NUM] = self.seed.loc[0, ALL_NUM].astype(float).values
        df.loc[0, "Internet_quality"] = self.seed.loc[0, "Internet_quality"]
        self.assertIn("exact copies", self.problems(df))

    def test_near_copy_caught(self):
        df = self.clean.copy()
        df[ALL_NUM] = df[ALL_NUM].astype(float)
        df.loc[1, ALL_NUM] = self.seed.loc[1, ALL_NUM].astype(float).values + 0.01
        self.assertIn("near-copies", self.problems(df))

    def test_email_caught(self):
        df = self.clean.copy()
        df.loc[4, "ID"] = "jane.doe@mail.com"
        self.assertIn("emails or phone numbers", self.problems(df))


suite = unittest.TestSuite()
for case in (TestGenerators, TestEvaluation, TestPrivacy):
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(case))
result = unittest.TextTestRunner(verbosity=2).run(suite)
print("\nALL TESTS PASSED" if result.wasSuccessful() else "\nSOME TESTS FAILED")

## 6. Decisions and limits (use this in your README)

- **Why this setting:** exam pass/fail has clear drivers, and it lines up with a related project.
- **Two generators on purpose:** the rule-based one encodes assumptions; the seed-based one learns from a few hand-made rows. Comparing them shows where each is weak.
- **Realism without real records:** checked through schema rules, designed patterns, model usefulness, and agreement between independent generators. No real student data is used anywhere.
- **No leakage:** `exam_score` is excluded from the model inputs because `Result` is calculated from it.
- **Reproducible:** every generator takes a seed.
- **Uncertainty:** cross-validation scores are reported as mean +/- spread.
- **Limits:** a small seed file gives noisy patterns; the seed-based generator samples `Internet_quality` independently of the score, so it loses that link; privacy checks reduce risk but cannot prove privacy.